In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
import rasterio
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.windows import from_bounds
from matplotlib.patches import Patch
import pyreadr

pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
# data load 
epa_stations = gpd.read_file('../01_data/01_raw/childs_pm/epa_station_locations/epa_station_locations.shp')
station_smoke = pyreadr.read_r('../01_data/01_raw/childs_pm/station_smokePM_2025_01.rds')[None]

In [ ]:
epa_stations.explore()

In [ ]:
epa_stations.head()

# grid_5km: 5km grid they constructed, they assigned each monitor to each grid cell and then defined the smoke for each epa station that way.
# shouldn't matter here.
# these are the 5km grid cell IDs. 

In [ ]:
station_smoke
station_smoke[station_smoke['id'] == '060371103'].head(25)

# smoke_day: whether there was a smoke plume intersecting the 5km grid cell for that station on that day.
# LA_wildfire_day: is it plausible that that grid cell is affected by the LA wildfires. this var is trying to determine whether the smoke that day at that station was due to LA wildfires. maybe just use this variable as a stratifier to look at, but prob don't have to use it in the analysis. just nice to know who is exposed to what.
# pm25_med_3yr: median pm2.5 on non-smoke days amongst all days for that station in that month and the 2 years prior.
    # pm25 - pm25_med_3yr = pm25_anom because they're looking at the anomalous smoke above the 3 yr median. and then the smoekPM var indicates whether it was a smoke day.
    # when smoke_day is 1, when there is a pm25_anom, that is attributed to smokePM and thus is the value in smokePM.
    # smokePM is noisy but not necessarily wrong. just may include some other anomalous PM but is close.
# smokePM: what is the PM2.5 that we think is from wf smoke? 
# light/med/dense is a measure of smoke.

# probably use the smokePM variable bc thats the smoke pm. it will be 0 when it was not a smoke day.

# NOTE: there are no stations in the palisades fire area. since the wind blew toward the water, there was just nothing to pick up on smoke in that area bc all stations are behind it. not a ton of people affected by the palisades fire smoke, many more from eaton. we corroborated this with the modis satellite data and where there are missing data from the satellite, there just aren't a lot of people

In [ ]:
# combining the data 
station_smoke_gdf = epa_stations.merge(station_smoke, left_on='stn_id', right_on='id', how='left')
station_smoke_gdf = gpd.GeoDataFrame(station_smoke_gdf, crs=station_smoke_gdf.crs)
station_smoke_gdf.explore()

In [ ]:
station_smoke_gdf.head()